In [1]:
!git clone https://github.com/Saba0033/nlp-final.git
%cd nlp-final

Cloning into 'nlp-final'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 46 (delta 12), reused 42 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 20.11 KiB | 10.06 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/nlp-final


### Generate the data for training

In [2]:
!ls
!ls data/

baseline     data     evaluate.py  report	     search
config.yaml  demo.py  model	   requirements.txt  train.py
analyze.py  __init__.py       preprocess.py	   preprocess_utils.py
dataset.py  preprocess_jm.py  preprocess_squad.py  preprocess_wiki.py


In [3]:
!python data/preprocess_squad.py

downlaoding squad...
README.md: 100% 7.62k/7.62k [00:00<00:00, 22.8MB/s]
plain_text/train-00000-of-00001.parquet: 100% 14.5M/14.5M [00:01<00:00, 11.1MB/s]
plain_text/validation-00000-of-00001.par(…): 100% 1.82M/1.82M [00:00<00:00, 5.03MB/s]
Generating train split: 100% 87599/87599 [00:00<00:00, 278340.82 examples/s]
Generating validation split: 100% 10570/10570 [00:00<00:00, 283589.26 examples/s]
  squad/train: 68536 pairs
  squad/val: 9155 pairs
  squad/test: 7880 pairs
squad done


In [4]:
!python data/preprocess_wiki.py

downlaoding wikipedia...
README.md: 100% 131k/131k [00:00<00:00, 153MB/s]
  wiki/train: 38070 pairs
  wiki/val: 4542 pairs
  wiki/test: 5136 pairs
wiki done


In [5]:
!ls data/processed/squad/
!ls data/processed/wiki/
!head -n 2 data/processed/squad/train.jsonl

test.jsonl  train.jsonl  val.jsonl
test.jsonl  train.jsonl  val.jsonl
{"query": "what three factors do scientists believe are the cause of sexual orientation?", "document": "scientists do not know the exact cause of sexual orientation, but they believe that it is caused by a complex interplay of genetic, hormonal, and environmental influences. they favor biologically-based theories, which point to genetic factors, the early uterine environment, both, or the inclusion of genetic and social factors. there is no substantive evidence which suggests parenting or early childhood experiences play a role when it comes to sexual orientation. research over several decades has demonstrated that sexual orientation ranges along a continuum, from exclusive attraction to the opposite sex to exclusive attraction to the same sex.", "doc_id": "570f8a7d5ab6b81900390f03", "source": "squad", "split": "train"}
{"query": "what is the favored influence as to the cause of sexual orientation?", "document": "sci

In [6]:
!python data/analyze.py


wiki
  train: {'count': 38070, 'unique_queries': 8870, 'unique_documents': 38020, 'dup_query_rate': 0.77, 'dup_doc_rate': 0.0, 'query_words_avg': 1.8, 'query_words_min_max': [1, 11], 'doc_words_avg': 73.1, 'doc_words_min_max': [40, 300]}
  val: {'count': 4542, 'unique_queries': 1108, 'unique_documents': 4541, 'dup_query_rate': 0.76, 'dup_doc_rate': 0.0, 'query_words_avg': 1.8, 'query_words_min_max': [1, 11], 'doc_words_avg': 72.9, 'doc_words_min_max': [40, 300]}
  test: {'count': 5136, 'unique_queries': 1109, 'unique_documents': 5136, 'dup_query_rate': 0.78, 'dup_doc_rate': 0.0, 'query_words_avg': 1.8, 'query_words_min_max': [1, 10], 'doc_words_avg': 72.9, 'doc_words_min_max': [40, 300]}
  train/test overlap: 0
  ex query: mimas (moon)
  ex doc  : mimas (from the greek μίμᾱς) is one of saturn's largest moons. it is also called saturn i. mimas is  ...
  ex query: mimas (moon)
  ex doc  : mimas has craters, but the most famous is the herschel crater. it is 130 km (80 mi) in diameter. th

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/nlp-final
!cp -r /content/nlp-final/data/processed /content/drive/MyDrive/nlp-final/

Mounted at /content/drive
cp: cannot stat '/content/nlp-final/data/processed': No such file or directory


In [8]:
!cp -r /content/nlp-final /content/drive/MyDrive/

In [30]:
%cd /content/drive/MyDrive/nlp-final

/content/drive/MyDrive/nlp-final


In [10]:
!ls

baseline     data	  logs			processed	  search
checkpoints  demo.py	  model			report		  train.py
config.yaml  evaluate.py  model_training.ipynb	requirements.txt


In [2]:
import os
print(os.getcwd())

/content


In [3]:
import json
import torch
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


## Data

We use the preprocessed SQuAD dataset, where each example is a (query, document) pair:
`query` is a natural-language question, and `document` is the passage containing the
answer. Train/val/test splits were generated by `preprocess_squad.py`.

In [4]:
def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [3]:


train = load_jsonl("/content/drive/MyDrive/nlp-final/data/processed/squad/train.jsonl")
val   = load_jsonl("/content/drive/MyDrive/nlp-final/data/processed/squad/val.jsonl")
test  = load_jsonl("/content/drive/MyDrive/nlp-final/data/processed/squad/test.jsonl")

print(f"train: {len(train)}")
print(f"val:   {len(val)}")
print(f"test:  {len(test)}")

train: 68536
val:   9155
test:  7880


In [4]:
sample = train[0]
print("keys:", list(sample.keys()))
print()
print("query:   ", sample["query"])
print()
print("document:", sample["document"][:300])

keys: ['query', 'document', 'doc_id', 'source', 'split']

query:    what three factors do scientists believe are the cause of sexual orientation?

document: scientists do not know the exact cause of sexual orientation, but they believe that it is caused by a complex interplay of genetic, hormonal, and environmental influences. they favor biologically-based theories, which point to genetic factors, the early uterine environment, both, or the inclusion of


In [15]:
query_lengths = [len(row["query"].split()) for row in train]
doc_lengths   = [len(row["document"].split()) for row in train]

print(f"query length  - avg: {sum(query_lengths)/len(query_lengths):.1f}, min: {min(query_lengths)}, max: {max(query_lengths)}")
print(f"doc length    - avg: {sum(doc_lengths)/len(doc_lengths):.1f},  min: {min(doc_lengths)},  max: {max(doc_lengths)}")
print(f"unique queries:   {len(set(row['query'] for row in train))}")
print(f"unique documents: {len(set(row['document'] for row in train))}")

query length  — avg: 10.0, min: 1, max: 40
doc length    — avg: 121.8,  min: 40,  max: 300
unique queries:   68322
unique documents: 14605


## Tokenizer

We use the pretrained `bert-base-uncased` tokenizer (vocab size 30,522) to convert text
into token IDs.

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"vocab size: {tokenizer.vocab_size}")

sample_text = "What causes sexual orientation?"
tokens = tokenizer(sample_text, return_tensors="pt")
print(f"input_ids shape: {tokens['input_ids'].shape}")
print(f"tokens: {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

vocab size: 30522
input_ids shape: torch.Size([1, 7])
tokens: ['[CLS]', 'what', 'causes', 'sexual', 'orientation', '?', '[SEP]']


In [17]:
tokens = tokenizer(sample_text, return_tensors="pt")
print("numbers:", tokens['input_ids'])
print("words:  ", tokenizer.convert_ids_to_tokens(tokens['input_ids'][0]))

numbers: tensor([[  101,  2054,  5320,  4424, 10296,  1029,   102]])
words:   ['[CLS]', 'what', 'causes', 'sexual', 'orientation', '?', '[SEP]']


## Dataset

`PairDataset` wraps the (query, document) pairs from our preprocessed train/val/test splits.
Each item is a tuple `(query, document)`, where `document` is the correct passage that
answers `query`. These positive pairs are used for contrastive training: during training,
other documents in the same batch serve as in-batch negatives.

In [5]:
from torch.utils.data import Dataset, DataLoader

class PairDataset(Dataset):
    def __init__(self, data):
        self.pairs = [(row["query"], row["document"]) for row in data]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]

In [8]:
X_train = PairDataset(train)
X_val   = PairDataset(val)
X_test  = PairDataset(test)

print(f"X_train: {len(X_train)} pairs")
print(f"X_val:   {len(X_val)} pairs")
print(f"X_test:  {len(X_test)} pairs")

query, doc = X_train[0]
print(f"\nquery: {query}")
print(f"doc:   {doc[:100]}...")

X_train: 68536 pairs
X_val:   9155 pairs
X_test:  7880 pairs

query: what three factors do scientists believe are the cause of sexual orientation?
doc:   scientists do not know the exact cause of sexual orientation, but they believe that it is caused by ...


In [9]:
BATCH_SIZE = 32

train_loader = DataLoader(X_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(X_val,   batch_size=BATCH_SIZE, shuffle=False)

print(f"train batches: {len(train_loader)}")
print(f"val batches: {len(val_loader)}")

queries, docs = next(iter(train_loader))
print(f"\nbatch size: {len(queries)}")
print(f"sample query: {queries[0]}")
print(f"sample doc:   {docs[0][:100]}...")

train batches: 2142
val batches: 287

batch size: 32
sample query: whose ambitions are credited with causing the conflict?
sample doc:   the franco-prussian war or franco-german war (german: deutsch-französischer krieg, lit. german-frenc...


## Model Hyperparameters

A small transformer encoder trained from scratch: 2 layers, 4 attention heads,
256-dim embeddings, projected down to a 128-dim output embedding for retrieval.

In [8]:
VOCAB_SIZE    = tokenizer.vocab_size
EMBED_DIM     = 256
NUM_HEADS     = 4
NUM_LAYERS    = 2
FFN_DIM       = 512
MAX_LEN       = 128
PROJECTION    = 128
DROPOUT       = 0.3

## Model Architecture

`TransformerEncoder` encodes a batch of texts into fixed-size embeddings:

1. Token + learned positional embeddings
2. A standard Transformer encoder stack (with padding mask)
3. Mean pooling over non-padding tokens
4. Linear projection to the final embedding dimension
5. L2 normalization, so embeddings can be compared via dot product (cosine similarity)

The same encoder is used for both queries and documents

In [9]:
import torch.nn as nn
import torch.nn.functional as F
import math

class TransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
        self.position_embedding = nn.Embedding(MAX_LEN, EMBED_DIM)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=EMBED_DIM,
            nhead=NUM_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=DROPOUT,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=NUM_LAYERS)


        self.projection = nn.Linear(EMBED_DIM, PROJECTION)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, input_ids, attention_mask):
        x = self.token_embedding(input_ids)

        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        x = x + self.position_embedding(positions)

        x = self.dropout(x)

        padding_mask = (attention_mask == 0)
        x = self.transformer(x, src_key_padding_mask=padding_mask)

        mask = attention_mask.unsqueeze(-1).float()
        x = (x * mask).sum(dim=1) / mask.sum(dim=1)
        x = self.projection(x)

        return F.normalize(x, dim=-1)

    def encode(self, texts):
        tokens = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        ).to(device)
        return self.forward(tokens["input_ids"], tokens["attention_mask"])

In [13]:
model = TransformerEncoder().to(device)
print(f"model parameters: {sum(p.numel() for p in model.parameters()):,}")

model parameters: 8,933,504


## Contrastive Loss (InfoNCE)

For a batch of (query, document) pairs, we treat each query's paired document as the
positive example and all other documents in the batch as negatives. InfoNCE computes
a softmax over similarity scores (scaled by temperature) and applies cross-entropy
against the diagonal (correct) pairing - pulling matching query/document embeddings
together and pushing non-matching pairs apart.

In [10]:
def infonce_loss(query_emb, doc_emb, temperature=0.05):
    logits = query_emb @ doc_emb.T / temperature
    labels = torch.arange(len(query_emb), device=query_emb.device)

    return F.cross_entropy(logits, labels)

In [11]:
q = torch.randn(4, PROJECTION)
d = torch.randn(4, PROJECTION)
q = F.normalize(q, dim=-1)
d = F.normalize(d, dim=-1)

loss = infonce_loss(q, d)
print(f"random embeddings loss: {loss.item():.4f}")
print(f"expected log(batch_size) = {math.log(32):.4f}")

random embeddings loss: 1.0038
expected log(batch_size) = 3.4657


## Training Setup (SQuAD)

We train with AdamW, learning rate 3e-4, batch size 32, for 10 epochs. Training and
validation loss are logged to TensorBoard, and the model checkpoint with the lowest
validation loss is saved.

In [51]:
from torch.utils.tensorboard import SummaryWriter
import os

EPOCHS = 10
LR = 3e-4

os.makedirs("/content/drive/MyDrive/nlp-final/checkpoints/squad_v2", exist_ok=True)
os.makedirs("/content/drive/MyDrive/nlp-final/logs/squad_v2", exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
writer = SummaryWriter("/content/drive/MyDrive/nlp-final/logs/squad_v2")

best_val_loss = float("inf")
global_step = 0

## Training and Validation Loops

`train_epoch` runs one pass over the training data, computing InfoNCE loss on each
batch and updating model weights. `val_epoch` evaluates the same loss on the
validation set without updating weights, used to track generalization and select
the best checkpoint.

In [22]:
def train_epoch(epoch):
    global global_step
    model.train()
    total_loss = 0

    for i, (queries, docs) in enumerate(train_loader):
        q_emb = model.encode(list(queries))
        d_emb = model.encode(list(docs))

        loss = infonce_loss(q_emb, d_emb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        writer.add_scalar("loss/train_step", loss.item(), global_step)
        global_step += 1

        if i % 100 == 0:
            print(f"  step {i}/{len(train_loader)} | loss {loss.item():.4f}")

    return total_loss / len(train_loader)

In [23]:
def val_epoch():
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for queries, docs in val_loader:
            q_emb = model.encode(list(queries))
            d_emb = model.encode(list(docs))
            total_loss += infonce_loss(q_emb, d_emb).item()

    return total_loss / len(val_loader)

## Run Training (SQuAD)

Trains for 10 epochs, saving the model whenever validation loss improves.

In [54]:
for epoch in range(EPOCHS):
    print(f"\nepoch {epoch+1}/{EPOCHS}")

    train_loss = train_epoch(epoch)
    val_loss   = val_epoch()

    writer.add_scalar("loss/train_epoch", train_loss, epoch)
    writer.add_scalar("loss/val_epoch",   val_loss,   epoch)

    print(f"train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/content/drive/MyDrive/nlp-final/checkpoints/squad_v2/best_model.pt")
        print(f"saved best model")

writer.close()
print("\ntraining done")


epoch 1/10
  step 0/2142 | loss 3.3840
  step 100/2142 | loss 2.9334
  step 200/2142 | loss 2.8444
  step 300/2142 | loss 3.0342
  step 400/2142 | loss 3.1235
  step 500/2142 | loss 2.7592
  step 600/2142 | loss 2.9079
  step 700/2142 | loss 2.6614
  step 800/2142 | loss 2.7222
  step 900/2142 | loss 2.7012
  step 1000/2142 | loss 3.0810
  step 1100/2142 | loss 2.4690
  step 1200/2142 | loss 2.5768
  step 1300/2142 | loss 2.4665
  step 1400/2142 | loss 2.7693
  step 1500/2142 | loss 2.4850
  step 1600/2142 | loss 2.7689
  step 1700/2142 | loss 2.5809
  step 1800/2142 | loss 2.5872
  step 1900/2142 | loss 2.5994
  step 2000/2142 | loss 2.7648
  step 2100/2142 | loss 2.1084
train loss: 2.7591 | val loss: 2.9022
saved best model

epoch 2/10
  step 0/2142 | loss 2.3053
  step 100/2142 | loss 2.6305
  step 200/2142 | loss 2.0297
  step 300/2142 | loss 2.4521
  step 400/2142 | loss 2.1833
  step 500/2142 | loss 2.1952
  step 600/2142 | loss 2.3200
  step 700/2142 | loss 1.3308
  step 800/21

Architecture:
  VOCAB_SIZE    = 30522
  EMBED_DIM     = 256
  NUM_HEADS     = 4
  NUM_LAYERS    = 2
  FFN_DIM       = 512
  MAX_LEN       = 128
  PROJECTION    = 128
  DROPOUT       = 0.3

Training:
  EPOCHS        = 10
  LR            = 3e-4
  BATCH_SIZE    = 32
  TEMPERATURE   = 0.05
  OPTIMIZER     = AdamW
  LOSS          = InfoNCE

Results (train | val):
  epoch 1:  2.7591 | 2.9022
  epoch 2:  2.0957 | 2.8134
  epoch 3:  1.6771 | 2.7790
  epoch 4:  1.4206 | 2.8069
  epoch 5:  1.2345 | 2.7474
  epoch 6:  1.0983 | 2.7339
  epoch 7:  0.9838 | 2.7137
  epoch 8:  0.8924 | 2.7225
  epoch 9:  0.8023 | 2.7315
  epoch 10: 0.7348 | 2.7040   <- best val, saved checkpoint (squad_v2)

Model parameters: 8,933,504
Dataset: SQuAD (68,536 train pairs)

Conclusion: clear overfitting -- train loss drops from 2.76 to 0.73
while val loss stays around 2.7, barely moving.

## SQuAD Training Results

Run summary (architecture, hyperparameters, and per-epoch losses) below.

In [17]:
model.load_state_dict(torch.load(
    "/content/drive/MyDrive/nlp-final/checkpoints/squad_v2/best_model.pt",
    map_location=device
))
model.eval()

import random
samples = random.sample(val, 5)
with torch.no_grad():
    for sample in samples:
        query = sample["query"]
        correct_doc = sample["document"]

        q_emb = model.encode([query])
        d_emb = model.encode([correct_doc])

        similarity = (q_emb @ d_emb.T).item()
        print(f"query: {query[:60]}")
        print(f"similarity with correct doc: {similarity:.4f}")
        print()

query: how much money did taxpayers provide in government support t
similarity with correct doc: 0.6692

query: who developed the dynagroove format?
similarity with correct doc: 0.3619

query: what is a choice that users of pesticides can make that will
similarity with correct doc: 0.5979

query: what includes topics like bitwise encoding, collation, and r
similarity with correct doc: 0.4414

query: which language did ljudevit gaj and vuk karadzic prefer?
similarity with correct doc: 0.4977



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


In [18]:
with torch.no_grad():
    for sample in random.sample(val, 3):
        query = sample["query"]
        correct_doc = sample["document"]


        wrong_docs = [random.choice(val)["document"] for _ in range(4)]
        all_docs = [correct_doc] + wrong_docs

        q_emb = model.encode([query])
        d_emb = model.encode(all_docs)

        scores = (q_emb @ d_emb.T).squeeze()
        ranked = scores.argsort(descending=True).tolist()

        print(f"query: {query[:60]}")
        print(f"correct doc rank: {ranked.index(0) + 1} out of 5")
        print(f"scores: {[round(scores[i].item(), 3) for i in ranked]}")
        print()

query: who did everton enter talks with to build a new 55,000 seat 
correct doc rank: 1 out of 5
scores: [0.587, 0.16, 0.091, -0.047, -0.101]

query: the children of france were given a catechism that taught th
correct doc rank: 1 out of 5
scores: [0.319, 0.256, 0.056, -0.059, -0.127]

query: what airport is located in east boston?
correct doc rank: 1 out of 5
scores: [0.909, 0.322, 0.263, 0.228, 0.009]



## Wiki Dataset - Second Training Run

We repeat the same training procedure on a Wikipedia-derived dataset, where each
query is a short title/phrase and the document is the corresponding section/paragraph.
We reinitialize the model with fresh weights to
train a separate model for this dataset.

In [25]:
BATCH_SIZE = 32

train_wiki = load_jsonl("/content/drive/MyDrive/nlp-final/data/processed/wiki/train.jsonl")
val_wiki   = load_jsonl("/content/drive/MyDrive/nlp-final/data/processed/wiki/val.jsonl")
test_wiki  = load_jsonl("/content/drive/MyDrive/nlp-final/data/processed/wiki/test.jsonl")

X_train_wiki = PairDataset(train_wiki)
X_val_wiki   = PairDataset(val_wiki)

train_loader = DataLoader(X_train_wiki, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(X_val_wiki,   batch_size=BATCH_SIZE, shuffle=False)

print(f"wiki train: {len(X_train_wiki)}")
print(f"wiki val:   {len(X_val_wiki)}")

wiki train: 38070
wiki val:   4542


In [26]:
model = TransformerEncoder().to(device)
print(f"model parameters: {sum(p.numel() for p in model.parameters()):,}")

model parameters: 8,933,504


## Training Setup (Wiki)

Same setup as SQuAD, but with a lower learning rate (1e-4 vs 3e-4). Wiki queries are
much shorter (often 1-2 words, for example: article titles), which makes the contrastive
task harder and more prone to overfitting - the lower LR is an attempt to fix this.

In [27]:
from torch.utils.tensorboard import SummaryWriter
import os

EPOCHS = 10
LR = 1e-4

os.makedirs("/content/drive/MyDrive/nlp-final/checkpoints/wiki", exist_ok=True)
os.makedirs("/content/drive/MyDrive/nlp-final/logs/wiki", exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
writer = SummaryWriter("/content/drive/MyDrive/nlp-final/logs/wiki")

best_val_loss = float("inf")
global_step = 0

## Run Training (Wiki)

**Result:** Validation loss increased across all 10 epochs (e.g. 3.78 ->
5.07) despite the lower learning rate, while training loss decreased -
indicating overfitting. We attribute this to the short, ambiguous nature of
Wiki queries (often single words or short titles), which provide insufficient signal
for the contrastive objective to generalize. We therefore do not use the Wiki model
for downstream evaluation or the demo, and report only the SQuAD model's results.

In [28]:
for epoch in range(EPOCHS):
    print(f"\nepoch {epoch+1}/{EPOCHS}")

    train_loss = train_epoch(epoch)
    val_loss   = val_epoch()

    writer.add_scalar("loss/train_epoch", train_loss, epoch)
    writer.add_scalar("loss/val_epoch",   val_loss,   epoch)

    print(f"train loss: {train_loss:.4f} | val loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/content/drive/MyDrive/nlp-final/checkpoints/wiki/best_model.pt")
        print(f"saved best model")

writer.close()
print("\ntraining done")


epoch 1/10
  step 0/1190 | loss 3.4570
  step 100/1190 | loss 3.2715
  step 200/1190 | loss 3.2156
  step 300/1190 | loss 2.9052
  step 400/1190 | loss 2.8910
  step 500/1190 | loss 3.2342
  step 600/1190 | loss 3.0748
  step 700/1190 | loss 3.3521
  step 800/1190 | loss 3.1922
  step 900/1190 | loss 2.2734
  step 1000/1190 | loss 3.0928
  step 1100/1190 | loss 2.7606
train loss: 3.0719 | val loss: 3.7758
saved best model

epoch 2/10
  step 0/1190 | loss 2.8333
  step 100/1190 | loss 2.9239
  step 200/1190 | loss 2.7764
  step 300/1190 | loss 2.7060
  step 400/1190 | loss 2.4950
  step 500/1190 | loss 2.5061
  step 600/1190 | loss 2.2826
  step 700/1190 | loss 2.4575
  step 800/1190 | loss 2.8321
  step 900/1190 | loss 2.1893
  step 1000/1190 | loss 2.5172
  step 1100/1190 | loss 2.6071
train loss: 2.6611 | val loss: 4.1432

epoch 3/10
  step 0/1190 | loss 2.1479
  step 100/1190 | loss 2.4926
  step 200/1190 | loss 2.1950
  step 300/1190 | loss 2.2466
  step 400/1190 | loss 2.5892
  s

## Wiki Data Format Check

Confirms the Wiki dataset has the same (query, document) structure as SQuAD -
here `query` is a short title/phrase rather than a full question.

In [29]:
sample = train_wiki[0]
print("keys:", list(sample.keys()))
print()
print("query:   ", sample["query"])
print()
print("document:", sample["document"][:300])

keys: ['query', 'document', 'doc_id', 'source', 'split']

query:    mimas (moon)

document: mimas (from the greek μίμᾱς) is one of saturn's largest moons. it is also called saturn i. mimas is best known for its large crater, herschel. in the centre of the crater is a very high mountain. mimas was discovered by the english astronomer william herschel on september 17, 1789. it resembles the 
